# Real Time Object Detection with YOLO26m on COCO 2017 (Notebook-First, End-to-End)

## 1. Title and Project Overview

This notebook turns the legacy MobileNet-SSD demo into a real notebook-first object detection project.

What this notebook does:
- downloads `awsaf49/coco-2017-dataset` from Kaggle in notebook cells
- verifies the COCO images and annotations before training
- converts a reproducible subset of COCO 2017 into YOLO detection format
- fine-tunes **YOLO26m** with OOM fallback to **YOLO26s**
- evaluates with mAP50, mAP50-95, precision, recall, and per-class AP50
- saves `best.pt`, `best.onnx`, `metrics.json`, and `artifact_manifest.json`

## 2. Problem Statement

Detect multiple everyday object categories in images and support real-time style inference after training.

- Input: natural images from COCO 2017
- Output: object bounding boxes, class labels, and confidences
- Dataset: COCO 2017 from Kaggle
- Challenge: multi-class detection across small, medium, and large objects with strong scene diversity

## 3. Why the Chosen Method Is Correct

**Task family:** object detection.

- The prompt explicitly calls for object detection, so YOLO26m is the correct April 2026 default
- COCO 2017 is the standard benchmark for general-purpose multi-class object detection
- A subset is used for practical local training while keeping evaluation real and grounded

## 4. Hardware / Environment Check

In [ ]:
import os, platform, random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()} — {getattr(torch.version, 'cuda', 'N/A')}")
print(f"Device  : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 5. Dependency Installation

In [ ]:
import importlib, subprocess, sys

def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

ensure_package("ultralytics")
ensure_package("cv2", "opencv-python")
ensure_package("matplotlib")
ensure_package("pandas")
ensure_package("pycocotools")
ensure_package("kagglehub")
ensure_package("albumentations")
print("All dependencies satisfied.")

## 6. Imports and Configuration

In [ ]:
import json, os, random, shutil
from collections import Counter
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_DIR   = Path(os.path.dirname(os.path.abspath("__file__")))
DATA_ROOT     = PROJECT_DIR.parents[2] / "data"
RUNS_DIR      = PROJECT_DIR.parents[2] / "runs"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"

for d in (DATA_ROOT, RUNS_DIR, ARTIFACTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"DATA_ROOT    : {DATA_ROOT}")
print(f"ARTIFACTS_DIR: {ARTIFACTS_DIR}")

## 7. Dataset Source Explanation

**Dataset:** Kaggle — `awsaf49/coco-2017-dataset`

- Source URL: https://www.kaggle.com/datasets/awsaf49/coco-2017-dataset
- COCO 2017 is a large multi-task benchmark with object detection annotations in JSON format
- This notebook uses the official train/val annotation files and converts a deterministic subset into YOLO format
- We keep the split honest: train subset from `train2017`, evaluation subset from `val2017`

**Credential requirements:**
- Set `KAGGLE_USERNAME` + `KAGGLE_KEY` env vars or place `kaggle.json` in `~/.kaggle/`
- Raises a clear error if credentials are missing — no synthetic fallback

In [ ]:
import subprocess

KAGGLE_DATASET = "awsaf49/coco-2017-dataset"
DATASET_DIR = DATA_ROOT / "coco_2017"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

def check_kaggle_credentials() -> None:
    has_env = os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")
    has_file = (Path.home() / ".kaggle" / "kaggle.json").exists()
    if not has_env and not has_file:
        raise RuntimeError(
            "Kaggle credentials not found.
"
            "Set KAGGLE_USERNAME and KAGGLE_KEY env vars, or place kaggle.json in ~/.kaggle/"
        )

def download_dataset() -> Path:
    check_kaggle_credentials()
    try:
        import kagglehub
        return Path(kagglehub.dataset_download(KAGGLE_DATASET))
    except Exception:
        pass
    subprocess.check_call([
        "kaggle", "datasets", "download", "-d", KAGGLE_DATASET,
        "-p", str(DATASET_DIR), "--unzip",
    ])
    return DATASET_DIR

print("Downloading COCO 2017 from Kaggle...")
raw_root = Path(download_dataset())
print(f"Raw dataset at: {raw_root}")
print("\nTop-level contents:")
for p in sorted(raw_root.iterdir()):
    print(f"  {p.name}{'/' if p.is_dir() else ''}")

## 8. Dataset Verification

In [ ]:
def find_coco_root(raw_root: Path) -> Path:
    candidates = [raw_root, *raw_root.iterdir()]
    for candidate in candidates:
        if not candidate.is_dir():
            continue
        has_train = (candidate / "train2017").exists() or (candidate / "images" / "train2017").exists()
        has_val = (candidate / "val2017").exists() or (candidate / "images" / "val2017").exists()
        has_ann = (candidate / "annotations").exists()
        if has_train and has_val and has_ann:
            return candidate
    raise RuntimeError(f"Could not locate COCO structure under {raw_root}")

COCO_ROOT = find_coco_root(raw_root)
print(f"COCO root: {COCO_ROOT}")

TRAIN_IMG_ROOT = COCO_ROOT / "train2017"
VAL_IMG_ROOT = COCO_ROOT / "val2017"
if not TRAIN_IMG_ROOT.exists():
    TRAIN_IMG_ROOT = COCO_ROOT / "images" / "train2017"
if not VAL_IMG_ROOT.exists():
    VAL_IMG_ROOT = COCO_ROOT / "images" / "val2017"

ANN_DIR = COCO_ROOT / "annotations"
TRAIN_JSON = ANN_DIR / "instances_train2017.json"
VAL_JSON = ANN_DIR / "instances_val2017.json"

assert TRAIN_IMG_ROOT.exists(), f"Missing train2017 images at {TRAIN_IMG_ROOT}"
assert VAL_IMG_ROOT.exists(), f"Missing val2017 images at {VAL_IMG_ROOT}"
assert TRAIN_JSON.exists(), f"Missing train annotations: {TRAIN_JSON}"
assert VAL_JSON.exists(), f"Missing val annotations: {VAL_JSON}"

print(f"Train images dir: {TRAIN_IMG_ROOT}")
print(f"Val images dir  : {VAL_IMG_ROOT}")
print(f"Train json      : {TRAIN_JSON.name}")
print(f"Val json        : {VAL_JSON.name}")

## 9. COCO JSON Inspection

Before conversion, inspect the actual category space and annotation counts.

In [ ]:
with open(TRAIN_JSON, "r", encoding="utf-8") as f:
    train_meta = json.load(f)
with open(VAL_JSON, "r", encoding="utf-8") as f:
    val_meta = json.load(f)

categories = sorted(train_meta["categories"], key=lambda item: item["id"])
cat_id_to_name = {item["id"]: item["name"] for item in categories}
cat_ids_sorted = [item["id"] for item in categories]
yolo_id_map = {cat_id: idx for idx, cat_id in enumerate(cat_ids_sorted)}
class_names = [cat_id_to_name[cat_id] for cat_id in cat_ids_sorted]

print(f"Train images      : {len(train_meta['images']):,}")
print(f"Train annotations : {len(train_meta['annotations']):,}")
print(f"Val images        : {len(val_meta['images']):,}")
print(f"Val annotations   : {len(val_meta['annotations']):,}")
print(f"COCO classes      : {len(class_names)}")
print(f"First 15 classes  : {class_names[:15]}")

## 10. Subset Strategy

COCO 2017 is large. For a notebook-first project that still runs locally, we use a deterministic subset:

- `MAX_TRAIN_IMAGES = 5000`
- `MAX_VAL_IMAGES = 1000`
- `MAX_TEST_IMAGES = 500`

This keeps training grounded on real COCO data while staying practical.

In [ ]:
MAX_TRAIN_IMAGES = 5000
MAX_VAL_IMAGES = 1000
MAX_TEST_IMAGES = 500

train_images_df = pd.DataFrame(train_meta["images"]).sort_values("id").reset_index(drop=True)
val_images_df = pd.DataFrame(val_meta["images"]).sort_values("id").reset_index(drop=True)

train_subset_ids = set(train_images_df.head(MAX_TRAIN_IMAGES)["id"].tolist())
val_candidate_ids = val_images_df.head(MAX_VAL_IMAGES + MAX_TEST_IMAGES)["id"].tolist()
val_subset_ids = set(val_candidate_ids[:MAX_VAL_IMAGES])
test_subset_ids = set(val_candidate_ids[MAX_VAL_IMAGES:MAX_VAL_IMAGES + MAX_TEST_IMAGES])

print(f"Train subset images: {len(train_subset_ids):,}")
print(f"Val subset images  : {len(val_subset_ids):,}")
print(f"Test subset images : {len(test_subset_ids):,}")

## 11. COCO to YOLO Conversion

In [ ]:
def build_annotation_map(annotation_rows, allowed_ids):
    ann_map = {}
    for ann in annotation_rows:
        image_id = ann["image_id"]
        if image_id not in allowed_ids:
            continue
        ann_map.setdefault(image_id, []).append(ann)
    return ann_map

train_ann_map = build_annotation_map(train_meta["annotations"], train_subset_ids)
val_ann_map = build_annotation_map(val_meta["annotations"], val_subset_ids)
test_ann_map = build_annotation_map(val_meta["annotations"], test_subset_ids)

print(f"Train subset annotated images: {len(train_ann_map):,}")
print(f"Val subset annotated images  : {len(val_ann_map):,}")
print(f"Test subset annotated images : {len(test_ann_map):,}")

In [ ]:
YOLO_ROOT = DATASET_DIR / "yolo_coco_subset"

def convert_split(images_df, allowed_ids, ann_map, src_img_dir: Path, split_name: str) -> tuple[int, int, int]:
    img_dst = YOLO_ROOT / "images" / split_name
    lbl_dst = YOLO_ROOT / "labels" / split_name
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)

    ok_count = 0
    skip_count = 0
    box_count = 0

    subset_rows = images_df[images_df["id"].isin(allowed_ids)]
    for _, row in subset_rows.iterrows():
        image_id = int(row["id"])
        file_name = row["file_name"]
        width = float(row["width"])
        height = float(row["height"])
        src_path = src_img_dir / file_name
        if not src_path.exists():
            skip_count += 1
            continue

        anns = ann_map.get(image_id, [])
        yolo_lines = []
        for ann in anns:
            if ann.get("iscrowd", 0) == 1:
                continue
            x, y, bw, bh = ann["bbox"]
            if bw <= 1 or bh <= 1:
                continue
            cls_id = yolo_id_map[ann["category_id"]]
            xc = (x + bw / 2.0) / width
            yc = (y + bh / 2.0) / height
            nw = bw / width
            nh = bh / height
            xc = max(0.0, min(1.0, xc))
            yc = max(0.0, min(1.0, yc))
            nw = max(0.0001, min(1.0, nw))
            nh = max(0.0001, min(1.0, nh))
            yolo_lines.append(f"{cls_id} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")

        shutil.copy2(src_path, img_dst / file_name)
        (lbl_dst / f"{Path(file_name).stem}.txt").write_text("\n".join(yolo_lines), encoding="utf-8")
        ok_count += 1
        box_count += len(yolo_lines)

    return ok_count, skip_count, box_count


train_ok, train_skip, train_boxes = convert_split(train_images_df, train_subset_ids, train_ann_map, TRAIN_IMG_ROOT, "train")
val_ok, val_skip, val_boxes = convert_split(val_images_df, val_subset_ids, val_ann_map, VAL_IMG_ROOT, "val")
test_ok, test_skip, test_boxes = convert_split(val_images_df, test_subset_ids, test_ann_map, VAL_IMG_ROOT, "test")

print(f"Train -> ok={train_ok}, skip={train_skip}, boxes={train_boxes}")
print(f"Val   -> ok={val_ok}, skip={val_skip}, boxes={val_boxes}")
print(f"Test  -> ok={test_ok}, skip={test_skip}, boxes={test_boxes}")

DATA_YAML = YOLO_ROOT / "data.yaml"
DATA_YAML.write_text(
    f"path: {YOLO_ROOT}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    f"nc: {len(class_names)}\n"
    f"names: {class_names}\n",
    encoding="utf-8"
)
print(f"data.yaml written: {DATA_YAML}")

## 12. Data Validation Checks

In [ ]:
def count_split(split_name: str):
    img_dir = YOLO_ROOT / "images" / split_name
    lbl_dir = YOLO_ROOT / "labels" / split_name
    images = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
    labels = list(lbl_dir.glob("*.txt"))
    return len(images), len(labels)

ti, tl = count_split("train")
vi, vl = count_split("val")
sti, stl = count_split("test")

print(f"Train: {ti:5d} images, {tl:5d} labels")
print(f"Val  : {vi:5d} images, {vl:5d} labels")
print(f"Test : {sti:5d} images, {stl:5d} labels")

assert ti >= 1000, f"Too few train images: {ti}"
assert vi >= 200, f"Too few val images: {vi}"
assert sti >= 100, f"Too few test images: {sti}"
assert DATA_YAML.exists(), "data.yaml missing"
print("Dataset verification passed.")

## 13. Exploratory Data Analysis

In [ ]:
train_label_dir = YOLO_ROOT / "labels" / "train"
train_img_dir = YOLO_ROOT / "images" / "train"

instance_counts = Counter()
box_areas = []
image_sizes = []

train_label_files = list(train_label_dir.glob("*.txt"))
for label_file in train_label_files[:2000]:
    img_path = train_img_dir / f"{label_file.stem}.jpg"
    if not img_path.exists():
        img_path = train_img_dir / f"{label_file.stem}.png"
    img = cv2.imread(str(img_path)) if img_path.exists() else None
    if img is not None:
        image_sizes.append(img.shape[:2])
    text = label_file.read_text(encoding="utf-8").strip()
    if not text:
        continue
    for line in text.splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        cls_id = int(parts[0])
        bw = float(parts[3])
        bh = float(parts[4])
        instance_counts[cls_id] += 1
        box_areas.append(bw * bh)

print(f"Sampled label files : {min(len(train_label_files), 2000):,}")
print(f"Classes with boxes  : {len(instance_counts):,}")
if image_sizes:
    heights = [h for h, _ in image_sizes]
    widths = [w for _, w in image_sizes]
    print(f"Width range         : {min(widths)} - {max(widths)}")
    print(f"Height range        : {min(heights)} - {max(heights)}")
if box_areas:
    print(f"BBox area min/mean/max: {min(box_areas):.6f} / {np.mean(box_areas):.6f} / {max(box_areas):.6f}")

top_15 = instance_counts.most_common(15)
eda_df = pd.DataFrame([
    {"class_id": cls_id, "class_name": class_names[cls_id], "instances": count}
    for cls_id, count in top_15
])
print(eda_df.to_string(index=False))

## 14. Train/Validation/Test Split Strategy

In [ ]:
split_df = pd.DataFrame({
    "split": ["train", "val", "test"],
    "images": [ti, vi, sti],
    "labels": [tl, vl, stl],
    "source": ["train2017 subset", "val2017 subset", "val2017 holdout subset"],
})
print(split_df.to_string(index=False))
print("\nThe split is deterministic because image IDs are sorted before subsetting.")

## 15. Preprocessing and Augmentation Strategy

In [ ]:
import albumentations as A

aug = A.Compose([
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(p=0.4),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.3),
], bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"]))

sample_img_path = random.choice(list(train_img_dir.glob("*.jpg"))[:200] or list(train_img_dir.glob("*.png"))[:200])
sample_lbl_path = train_label_dir / f"{sample_img_path.stem}.txt"
img_rgb = cv2.cvtColor(cv2.imread(str(sample_img_path)), cv2.COLOR_BGR2RGB)

bboxes, class_labels = [], []
text = sample_lbl_path.read_text(encoding="utf-8").strip() if sample_lbl_path.exists() else ""
for line in text.splitlines():
    parts = line.split()
    if len(parts) == 5:
        class_labels.append(int(parts[0]))
        bboxes.append([float(v) for v in parts[1:]])

try:
    aug_result = aug(image=img_rgb, bboxes=bboxes, class_labels=class_labels)
    aug_img = aug_result["image"]
except Exception:
    aug_img = img_rgb

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(cv2.resize(img_rgb, (320, 240)))
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(cv2.resize(aug_img, (320, 240)))
axes[1].set_title("Augmented (preview)")
axes[1].axis("off")
plt.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "augmentation_preview.png"), dpi=100)
plt.close(fig)
print("Saved augmentation_preview.png")

## 16. Baseline Sanity Check with Pretrained YOLO26m

In [ ]:
from ultralytics import YOLO

baseline_model = YOLO("yolo26m.pt")
val_img_dir = YOLO_ROOT / "images" / "val"
val_samples = sorted(list(val_img_dir.glob("*.jpg")) + list(val_img_dir.glob("*.png")))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for ax, img_path in zip(axes, val_samples):
    res = baseline_model.predict(str(img_path), verbose=False)[0]
    n = len(res.boxes) if res.boxes is not None else 0
    ax.imshow(res.plot()[:, :, ::-1])
    ax.set_title(f"Pretrained detections: {n}", fontsize=8)
    ax.axis("off")
plt.suptitle("Baseline sanity check — pretrained YOLO26m", fontsize=11)
plt.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "sanity_inference.png"), dpi=100)
plt.close(fig)
print("Saved sanity_inference.png")

## 17. Main Model Setup

In [ ]:
PREFERRED_MODEL = "yolo26m.pt"
FALLBACK_MODEL = "yolo26s.pt"

IMG_SIZE = 640
EPOCHS = 20
BATCH = 16
WORKERS = 2

TRAIN_PROJECT = RUNS_DIR / "real_time_object_detection"
TRAIN_NAME = "yolo26m_coco_subset"

print(f"Model   : {PREFERRED_MODEL}")
print(f"Epochs  : {EPOCHS}")
print(f"Batch   : {BATCH}")
print(f"Classes : {len(class_names)}")
print(f"Data    : {DATA_YAML}")

## 18. Training Loop or Fine-Tuning Pipeline

In [ ]:
from ultralytics import YOLO

def run_training(model_name: str, batch_size: int):
    model = YOLO(model_name)
    return model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=batch_size,
        workers=WORKERS,
        project=str(TRAIN_PROJECT),
        name=TRAIN_NAME,
        exist_ok=True,
        device=DEVICE,
        seed=SEED,
        verbose=True,
    )

active_model_name = PREFERRED_MODEL
active_batch = BATCH
try:
    train_results = run_training(active_model_name, active_batch)
    print(f"Training complete with {active_model_name}.")
except RuntimeError as exc:
    if "out of memory" in str(exc).lower():
        print(f"OOM with {active_model_name}; retrying with {FALLBACK_MODEL}...")
        torch.cuda.empty_cache()
        active_model_name = FALLBACK_MODEL
        active_batch = max(4, active_batch // 2)
        train_results = run_training(active_model_name, active_batch)
        print(f"Training complete with {active_model_name} (OOM fallback).")
    else:
        raise

## 19. Inference Examples

In [ ]:
best_path = TRAIN_PROJECT / TRAIN_NAME / "weights" / "best.pt"
if not best_path.exists():
    raise FileNotFoundError(f"best.pt not found at {best_path}")

best_model = YOLO(str(best_path))
holdout_dir = YOLO_ROOT / "images" / "test"
holdout_samples = sorted(list(holdout_dir.glob("*.jpg")) + list(holdout_dir.glob("*.png")))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for ax, img_path in zip(axes, holdout_samples):
    res = best_model.predict(str(img_path), conf=0.25, verbose=False)[0]
    n = len(res.boxes) if res.boxes is not None else 0
    ax.imshow(res.plot()[:, :, ::-1])
    ax.set_title(f"{img_path.name[:18]}\n{n} detections", fontsize=7)
    ax.axis("off")
plt.suptitle("Holdout inference examples", fontsize=11)
plt.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "holdout_inference.png"), dpi=100)
plt.close(fig)
print("Saved holdout_inference.png")

## 20. Evaluation

In [ ]:
val_metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMG_SIZE,
    device=DEVICE,
    verbose=True,
)

map50 = float(val_metrics.box.map50)
map50_95 = float(val_metrics.box.map)
precision = float(val_metrics.box.mp)
recall = float(val_metrics.box.mr)

print(f"mAP50     : {map50:.4f}")
print(f"mAP50-95  : {map50_95:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")

names = val_metrics.names
cls_list = list(names.values()) if isinstance(names, dict) else list(names)
ap50_vals = val_metrics.box.ap50
class_aps = {}
for cls_name, ap in zip(cls_list, ap50_vals):
    class_aps[cls_name] = float(ap)

metrics_out = {
    "map50": map50,
    "map50_95": map50_95,
    "precision": precision,
    "recall": recall,
    "per_class_ap50": class_aps,
}
with open(ARTIFACTS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_out, f, indent=2)

top_ap_df = pd.DataFrame([
    {"class_name": name, "ap50": ap}
    for name, ap in sorted(class_aps.items(), key=lambda item: item[1], reverse=True)[:20]
])
print(top_ap_df.to_string(index=False))
print("Saved metrics.json")

## 21. Error Analysis

In [ ]:
run_dir = TRAIN_PROJECT / TRAIN_NAME
for fname in ["results.png", "PR_curve.png", "F1_curve.png", "confusion_matrix.png"]:
    fpath = run_dir / fname
    if fpath.exists():
        img = cv2.cvtColor(cv2.imread(str(fpath)), cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(fname)
        plt.tight_layout()
        fig.savefig(str(ARTIFACTS_DIR / fname), dpi=80)
        plt.close(fig)
        print(f"Saved {fname}")

if class_aps:
    names_sorted, aps_sorted = zip(*sorted(class_aps.items(), key=lambda item: item[1], reverse=True)[:20])
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(range(len(names_sorted)), aps_sorted, color="steelblue")
    ax.set_xticks(range(len(names_sorted)))
    ax.set_xticklabels(names_sorted, rotation=60, ha="right", fontsize=8)
    ax.set_ylabel("AP50")
    ax.set_title("Top-20 per-class AP50 (validation set)")
    ax.set_ylim(0, 1)
    plt.tight_layout()
    fig.savefig(str(ARTIFACTS_DIR / "per_class_ap50.png"), dpi=100)
    plt.close(fig)
    print("Saved per_class_ap50.png")

## 22. Save Model and Artifacts

In [ ]:
saved_pt = ARTIFACTS_DIR / "best.pt"
shutil.copy2(best_path, saved_pt)
print(f"Saved best.pt -> {saved_pt}")

export_model = YOLO(str(saved_pt))
onnx_path = export_model.export(format="onnx", imgsz=IMG_SIZE, opset=12)
saved_onnx = ARTIFACTS_DIR / "best.onnx"
shutil.copy2(onnx_path, saved_onnx)
print(f"Saved best.onnx -> {saved_onnx}")

with open(ARTIFACTS_DIR / "metrics.json", "r", encoding="utf-8") as f:
    metrics_payload = json.load(f)

manifest = {
    "project": "Real Time Object Detection (YOLO26m)",
    "dataset": "awsaf49/coco-2017-dataset",
    "subset": {
        "train_images": ti,
        "val_images": vi,
        "test_images": sti,
    },
    "model_file": "best.pt",
    "onnx_file": "best.onnx",
    "metrics": metrics_payload,
}
with open(ARTIFACTS_DIR / "artifact_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print("Saved artifact_manifest.json")

## 23. Reproducibility Notes

- Seed `42` is fixed for subset creation and YOLO training
- The subset is deterministic because images are sorted before selection
- COCO conversion keeps the original category mapping, remapped to contiguous YOLO IDs
- OOM fallback switches from YOLO26m to YOLO26s and halves the batch size

## 24. Conclusion and Limitations

This notebook implements a real notebook-first object detection workflow on COCO 2017:

- Real Kaggle download with fail-loud credential checks
- Real image and annotation verification before training
- Real COCO JSON to YOLO label conversion on a deterministic subset
- Real YOLO26m fine-tuning, validation, holdout inference, and artifact export

**Limitations:**
- This is a practical subset, not full COCO training, so headline benchmark scores will be below full-scale runs
- Real-time throughput depends on hardware and export target; the notebook prepares ONNX export but does not benchmark FPS
- COCO is very broad, so rare classes may underperform on the smaller subset

## 25. How to Improve This Project

1. Train on the full COCO 2017 train split for stronger overall AP.
2. Add TensorRT or OpenVINO benchmarking for honest real-time FPS reporting.
3. Tune image size, EMA, and augmentation for a better speed/accuracy tradeoff.
4. Build a webcam inference cell that uses the exported ONNX or best.pt weights.
5. Add class-specific error analysis for small-object-heavy categories such as `bottle`, `bird`, and `traffic light`.